# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/124pritivarma6001-commits/flyrank_internship_ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I chose a Decision Tree Classifier for the CTR / Engagement Opportunity Scoring lane.
The week-4 baseline uses threshold-based rules on impressions,CTR, and weighted average position. A decision tree is suitable because it can learnsimple threshold-based relationships and interactions between thees signals while remaining easy to interpret.

The model will be compared with the week-4 rule-based using the same data, metric and split. The goal is not to use a more complex model just for complexity but to check whether a simple interpreatble model cam provide a useful comparison to the existing baseline.

In [11]:
%pip -q install duckdb huggingface_hub

In [12]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [13]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [14]:
ctr_distribution = con.sql(f"""
WITH content_level AS (
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(
            CASE
                WHEN gsc_impressions > 0
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        ) / NULLIF(SUM(gsc_impressions), 0) AS avg_position
    FROM {TABLES['fact_daily']}
    WHERE gsc_impressions IS NOT NULL
      AND gsc_clicks IS NOT NULL
      AND gsc_avg_position IS NOT NULL
    GROUP BY content_hash_id
),
base AS (
    SELECT
        *,
        CAST(clicks AS DOUBLE) / NULLIF(impressions, 0) AS ctr
    FROM content_level
    WHERE impressions > 0
)
SELECT
    COUNT(*) AS total_contents,
    COUNT(*) FILTER (WHERE clicks > 0) AS contents_with_clicks,
    COUNT(*) FILTER (WHERE clicks = 0) AS contents_with_zero_clicks,
    MIN(ctr) AS min_ctr,
    MAX(ctr) AS max_ctr,
    QUANTILE_CONT(ctr, 0.25) AS ctr_25th,
    QUANTILE_CONT(ctr, 0.50) AS ctr_median,
    QUANTILE_CONT(ctr, 0.75) AS ctr_75th
FROM base
""").df()
ctr_distribution


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_contents,contents_with_clicks,contents_with_zero_clicks,min_ctr,max_ctr,ctr_25th,ctr_median,ctr_75th
0,309234,148941,160293,0.0,1.0,0.0,0.0,0.003024


In [15]:
ctr_threshold = con.sql(f"""
WITH content_level AS (
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN CAST(SUM(gsc_clicks) AS DOUBLE) / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr
    FROM {TABLES['fact_daily']}
    WHERE gsc_impressions IS NOT NULL
      AND gsc_clicks IS NOT NULL
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
)
SELECT
    QUANTILE_CONT(ctr, 0.75) AS ctr_threshold,
    QUANTILE_CONT(impressions, 0.50) AS impression_threshold
FROM content_level
WHERE ctr IS NOT NULL
""").df()

ctr_threshold

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ctr_threshold,impression_threshold
0,0.003026,237.0


In [16]:
rule_preview = con.sql(f"""
WITH content_level AS (
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN CAST(SUM(gsc_clicks) AS DOUBLE) / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr,
        SUM(gsc_avg_position * gsc_impressions)
        / NULLIF(SUM(gsc_impressions), 0) AS avg_position

    FROM {TABLES['fact_daily']}

    WHERE gsc_impressions IS NOT NULL
      AND gsc_clicks IS NOT NULL
      AND gsc_avg_position IS NOT NULL

    GROUP BY content_hash_id

    HAVING SUM(gsc_impressions) > 0
),

scored AS (
    SELECT
        *,
        CASE
            WHEN impressions >= 237
             AND ctr < 0.003026
             AND avg_position <= 20
            THEN 'OPPORTUNITY'
            ELSE 'NOT OPPORTUNITY'
        END AS rule_result,

        CASE
            WHEN impressions >= 237
             AND ctr < 0.003026
             AND avg_position <= 20
            THEN 'LOW_CTR_GOOD_POSITION'
            ELSE 'NOT_OPPORTUNITY'
        END AS reason_code

    FROM content_level
)

SELECT
    rule_result,
    reason_code,
    COUNT(*) AS rows_count
FROM scored
GROUP BY rule_result, reason_code
ORDER BY rule_result, reason_code
""").df()

rule_preview

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rule_result,reason_code,rows_count
0,NOT OPPORTUNITY,NOT_OPPORTUNITY,242472
1,OPPORTUNITY,LOW_CTR_GOOD_POSITION,66762


In [17]:
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42,
    class_weight="balanced",
)
model


DecisionTreeClassifier(class_weight='balanced', max_depth=4, random_state=42)

## 2. Split design

The Week-4 baseline was rule-based and did not use a train/test split. For Week 5, I use a fixed 80/20 content-level split so that the new model can be trained and evaluated consistently.

The split is performed after content-level aggregation, so each content item stays in only one partition. The test set is kept separate from model training and is used to compare the Decision Tree with the Week-4 rule-based baseline on the same rows.

A fixed random state is used so that the split is reproducible.

In [18]:
from sklearn.model_selection import train_test_split

# Build the content-level dataset used for Week 5
model_data = con.sql(f"""
WITH content_level AS (
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(
            CASE
                WHEN gsc_impressions > 0
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        ) / NULLIF(SUM(gsc_impressions), 0) AS avg_position
    FROM {TABLES['fact_daily']}
    WHERE gsc_impressions IS NOT NULL
      AND gsc_clicks IS NOT NULL
      AND gsc_avg_position IS NOT NULL
    GROUP BY content_hash_id
),
base AS (
    SELECT
        *,
        CAST(clicks AS DOUBLE) / NULLIF(impressions, 0) AS ctr
    FROM content_level
    WHERE impressions > 0
)
SELECT
    content_hash_id,
    impressions,
    clicks,
    ctr,
    avg_position,
    CASE
        WHEN impressions >= 237
         AND ctr < 0.003026
         AND avg_position <= 20
        THEN 1
        ELSE 0
    END AS opportunity
FROM base
WHERE ctr IS NOT NULL
  AND avg_position IS NOT NULL
""").df()

# Fixed 80/20 content-level split
train_data, test_data = train_test_split(
    model_data,
    test_size=0.20,
    random_state=42,
    stratify=model_data["opportunity"]
)

print("Total rows:", len(model_data))
print("Training rows:", len(train_data))
print("Test rows:", len(test_data))
print("Training opportunity rate:", train_data["opportunity"].mean())
print("Test opportunity rate:", test_data["opportunity"].mean())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows: 309234
Training rows: 247387
Test rows: 61847
Training opportunity rate: 0.2158965507484225
Test opportunity rate: 0.21588759357769982


## 3. Train + compare vs my baseline

I trained the Decision Tree on the training portion of the content-level dataset and evaluated it on the held-out test set.

For a fair comparison, the Week-4 rule-based baseline is also evaluated on the same test rows. The comparison uses accuracy, precision, recall, and F1 score.

Because the Week-4 opportunity label is defined directly from impressions, CTR, and average position, the model is mainly being tested on how well it reproduces the existing rule. Therefore, the results are interpreted as a baseline comparison rather than proof of real-world prediction quality.

In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# Features and target
features = [
    "impressions",
    "ctr",
    "avg_position"
]

X_train = train_data[features]
y_train = train_data["opportunity"]

X_test = test_data[features]
y_test = test_data["opportunity"]

# Train Decision Tree
model.fit(X_train, y_train)

# Decision Tree predictions
tree_pred = model.predict(X_test)

# Week-4 baseline predictions on the SAME test rows
baseline_pred = (
    (test_data["impressions"] >= 237)
    & (test_data["ctr"] < 0.003026)
    & (test_data["avg_position"] <= 20)
).astype(int)

# Calculate metrics
comparison = pd.DataFrame({
    "Model": [
        "Week-4 Rule Baseline",
        "Decision Tree"
    ],
    "Accuracy": [
        accuracy_score(y_test, baseline_pred),
        accuracy_score(y_test, tree_pred)
    ],
    "Precision": [
        precision_score(y_test, baseline_pred, zero_division=0),
        precision_score(y_test, tree_pred, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, baseline_pred, zero_division=0),
        recall_score(y_test, tree_pred, zero_division=0)
    ],
    "F1": [
        f1_score(y_test, baseline_pred, zero_division=0),
        f1_score(y_test, tree_pred, zero_division=0)
    ]
})

comparison.round(4)

,Model,Accuracy,Precision,Recall,F1
0,Week-4 Rule Baseline,1.0,1.0,1.0,1.0
1,Decision Tree,1.0,1.0,1.0,1.0


## 4. Errors and interpretation


The Decision Tree made very few errors on the held-out test set. The remaining errors are cases where the learned tree did not exactly reproduce the Week-4 rule.

I also inspected the feature importance to understand which signals the model relies on. Since the opportunity label is defined using impressions, CTR, and average position, high importance for these features is expected.

The results should be interpreted as agreement with the existing rule rather than independent evidence of real-world opportunity quality.

In [20]:
# Find model errors
error_mask = tree_pred != y_test.to_numpy()

errors = test_data.loc[
    error_mask,
    [
        "content_hash_id",
        "impressions",
        "clicks",
        "ctr",
        "avg_position",
        "opportunity"
    ]
].copy()

errors["tree_prediction"] = tree_pred[error_mask]

print("Number of Decision Tree errors:", len(errors))

if len(errors) > 0:
    display(errors.head(10))
else:
    print("No errors found on the held-out test set.")

# Feature importance
feature_importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("\nFeature importance:")
display(feature_importance)

Number of Decision Tree errors: 0
No errors found on the held-out test set.

Feature importance:


,feature,importance
0,impressions,0.467488
1,ctr,0.281344
2,avg_position,0.251168


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.